## EU REED

#### 1) Import packages and set filepath

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path
from reed import utils
from collections import defaultdict, Counter
from functools import reduce
import re

In [ ]:
config = utils.Config()
#test

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

#### 2) Import Monitoring data

In [ ]:
def parse_event_and_type_from_csv_filename(csv_filename: str):
    """
    CSV-filnavn (uden .csv) ser ud som fx:
      OF10001_G_TXG_REGRHM_92SQBY
    measurement_type = sidste 6 tegn: 92SQBY
    event_id = resten (uden evt. trailing underscore)
    """
    base = Path(csv_filename).name  # fjerner evt. sti inde i tar
    stem = base[:-4] if base.lower().endswith(".csv") else base

    if len(stem) < 6:
        return stem, None

    measurement_type = stem[-6:]
    event_part = stem[:-6]
    event_id = event_part[:-1] if event_part.endswith("_") else event_part
    return event_id, measurement_type


all_records = []

for company_folder in monitoring_data_path.iterdir():
    if not company_folder.is_dir():
        continue

    # Firmnavn = første del før " - "
    parts = [p.strip() for p in company_folder.name.split(" - ")]
    if len(parts) < 1:
        print("Advarsel – unexpected name of folder:", company_folder.name)
        continue

    company = parts[0].strip().lower()
    maskinenavn = " - ".join(parts[1:]) if len(parts) > 1 else None  # valgfrit, men nyttigt metadata

    # Find ALLE tar-filer i denne topmappe (inkl. undermapper for robusthed)
    tar_files = list(company_folder.rglob("*.tar")) + list(company_folder.rglob("*.tar.gz"))

    if not tar_files:
        print(f"Ingen .tar/.tar.gz-filer i {company_folder.name}")
        continue

    # Til dublet-advarsel: csv basenames indenfor denne company_folder
    seen_csv_basenames = []

    for tar_path in tar_files:
        try:
            # r:* autodetekterer .tar vs .tar.gz
            with tarfile.open(tar_path, mode="r:*") as tf:
                csv_members = [m for m in tf.getmembers() if m.name.lower().endswith(".csv")]

                if not csv_members:
                    print(f"Ingen CSV fundet i {tar_path.name}")
                    continue

                for csv_member in csv_members:
                    csv_basename = Path(csv_member.name).name
                    seen_csv_basenames.append(csv_basename)

                    event_id, measurement_type = parse_event_and_type_from_csv_filename(csv_basename)

                    f = tf.extractfile(csv_member)
                    if f is None:
                        print(f"Advarsel – kunne ikke extracte {csv_member.name} i {tar_path}")
                        continue

                    df = pd.read_csv(f)

                    all_records.append({
                        "company": company,
                        "maskinenavn": maskinenavn,
                        "event_id": event_id,
                        "measurement_type": measurement_type,
                        "tar_path": tar_path,
                        "csv_name": csv_member.name,     # sti inde i tar
                        "csv_basename": csv_basename,    # kun filnavn
                        "data": df,
                    })

        except tarfile.TarError as e:
            print(f"Advarsel – kunne ikke åbne arkiv: {tar_path} | {e}")

    # Dublet-melding for ens CSV-filnavne (basenames) indenfor company_folder
    c = Counter(seen_csv_basenames)
    dups = [name for name, cnt in c.items() if cnt > 1]
    if dups:
        print(f"\n⚠️ Dublet CSV-filnavne fundet i '{company_folder.name}':")
        for name in sorted(dups):
            print(f"  - {name} (x{c[name]})")

print("\nTotal amount of CSV imports:", len(all_records))

# === group company and event ===
events = defaultdict(list)
for rec in all_records:
    events[(rec["company"], rec["event_id"])].append(rec)

print("Events:", len(events))


#### 3) Import tool names and mapping to reed foreground processes

In [ ]:
def load_tool_operation_keys(tool_descriptions_path: Path):
    """
    Læser tool_operation_combined.xlsx fra tool_descriptions_path og returnerer:
      - key_company_tools: kombineret DF fra arkene KOMATSU, WWR, GOIMEK
          kolonner: company, Tool_Number, OP TYPE
      - key_reed_tools: DF fra arket REED (originale kolonnenavne)
    """
    xlsx_path = tool_descriptions_path / "tool_operation_combined.xlsx"

    # Ark der skal kombineres
    company_sheets = ["KOMATSU", "WWR", "GOIMEK"]

    dfs = []
    for sheet in company_sheets:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, skiprows=1)
        df_out = df[["TOOL", "OP TYPE"]].copy()
        df_out.insert(0, "company", sheet)
        df_out = df_out.rename(columns={"TOOL": "Tool_Number"})
        dfs.append(df_out)

        df_out["Tool_Number"] = (
            df_out["Tool_Number"]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .str.split("/")
        )
        df_out = df_out.explode("Tool_Number", ignore_index=True)
        df_out["Tool_Number"] = pd.to_numeric(df_out["Tool_Number"], errors="coerce")

        dfs.append(df_out)

    key_company_tools = pd.concat(dfs, ignore_index=True)

    # REED som separat df med originale kolonnenavne
    key_reed_tools = pd.read_excel(xlsx_path, sheet_name="REED")

    return key_company_tools, key_reed_tools

key_company_tools, key_reed_tools = load_tool_operation_keys(tool_descriptions_path)
key_company_tools["Tool_Number"] = pd.to_numeric(key_company_tools["Tool_Number"], errors="coerce")
key_company_tools["company"] = key_company_tools["company"].str.strip().str.lower()




#### 4) Merge all records and keys to final_df

In [ ]:
def merge_records(records):
    """
    Creating one dataFrame for each event.

    """
    dfs = []
    cols = ["timestamp", "Tool_Number", "Power_Active", "Energy_Total"]

    merged = None
    
    for i, r in enumerate(records):
        df = r["data"].copy()
        df = df[[c for c in cols if c in df.columns]].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.floor('s')
        df = df.set_index("timestamp")

        if merged is None:
            merged = df
        else:
            merged = merged.join(df, how="outer")

    merged = merged.sort_index()
            
    return merged

In [ ]:
all_grouped = []  
event_summaries = []
for (company, event_id), records in events.items():
    merged_df = merge_records(records)
    if merged_df.empty or "Power_Active" not in merged_df.columns:
        print("\n--- MISSING Power_Active ---")
        print("company:", company)
        print("event_id:", event_id)
        print("merged_df shape:", merged_df.shape)
        print("merged_df columns:", list(merged_df.columns))

        # extra: inspect what came in
        print("records type:", type(records), "len:", len(records) if hasattr(records, "__len__") else "n/a")
        if hasattr(records, "__len__") and len(records) > 0:
            r0 = records[0]
            print("first record type:", type(r0))
            if hasattr(r0, "columns"):
                print("first record columns:", list(r0.columns))
            elif isinstance(r0, dict):
                print("first record keys:", list(r0.keys()))
        # stop right here so you can inspect in VS Code
        breakpoint()

    merged_df["energy_consumption"] = merged_df["Power_Active"] / 3600

    merged_df = merged_df.reset_index()

    s = merged_df["Energy_Total"].dropna()

    if s.empty:
        meter_energy = np.nan
    else:
        meter_energy = s.iloc[-1] - s.iloc[0]

    event_summaries.append({
        "company": company,
        "event_id": event_id,
        "energy_meter_total": meter_energy,
    })

    def tool_hours(ts):
        ts = ts.sort_values()
        if ts.empty:
            return 0.0
        return (ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600.0

    # Group after tool_number
    grouped_df = (
        merged_df
        .groupby("Tool_Number", as_index=False)
        .agg(
            energy_consumption=("energy_consumption", "sum"),
            Power_Active=("Power_Active", "mean"),
            seconds=("timestamp", "count"),
            )
        )
    
    grouped_df["hours"] = grouped_df["seconds"] / 3600.0
    grouped_df["company"] = company
    grouped_df["event_id"] = event_id

    grouped_df = grouped_df.set_index(["company", "event_id", "Tool_Number"])

    all_grouped.append(grouped_df)

final_df = pd.concat(all_grouped)

# --- MERGE 1: tilføj OP TYPE via company + Tool_Number ---
final_df = final_df.reset_index()

final_df["Tool_Number"] = pd.to_numeric(final_df["Tool_Number"], errors="coerce")

final_df = final_df.merge(
    key_company_tools[["company", "Tool_Number", "OP TYPE"]],
    on=["company", "Tool_Number"],
    how="left"
)

# --- MERGE 2: tilføj REED-info via OP TYPE ---
final_df = final_df.merge(
    key_reed_tools,
    on="OP TYPE",
    how="left",
    suffixes=("", "_reed")
)

final_df["reed tool"] = final_df["reed tool"].fillna("machining, generic")
final_df = final_df.set_index(["company", "event_id", "Tool_Number"])


In [ ]:
print(final_df)

#### 5) Analyse which tool numbers are missing a name

In [ ]:
missing_op_type_tools = (
    final_df
    .loc[final_df["OP TYPE"].isna()]
    .reset_index()[["company", "Tool_Number"]]
    .drop_duplicates()
    .sort_values(["company", "Tool_Number"])
    .reset_index(drop=True)
)

missing_op_type_tools



#### 6) Split final_df into separate dataframes per event

In [ ]:
def split_final_df_by_company_event(final_df):
    """
    Splitter final_df (med MultiIndex: company, event_id, Tool_Number)
    til én dataframe pr. (company, event_id).

    Returnerer:
        dict[(company, event_id)] -> pd.DataFrame
    """
    dfs = {}

    for (company, event_id), df in final_df.groupby(level=["company", "event_id"]):
        dfs[(company, event_id)] = df.reset_index()

    return dfs

event_dfs = split_final_df_by_company_event(final_df)

# eksempel
print(event_dfs[("komatsu", "OF10006_G_SLG_E8AJ1C")])


#### 7) Ready for LCA import

In [53]:
def sum_by_reed_tool(
    df,
    reed_tool_col="reed tool",
    cols_to_sum=("energy_consumption", "hours", "seconds")):
    return (df
        .drop(columns=["Tool_Number"], errors="ignore")
        .groupby(reed_tool_col, as_index=False)
        .agg({col: "sum" for col in cols_to_sum}))

In [54]:
energy_consumption_by_LCA_process = sum_by_reed_tool(
    event_dfs[("komatsu", "OF10006_G_SLG_E8AJ1C")])

In [55]:
print(energy_consumption_by_LCA_process)

            reed tool  energy_consumption      hours  seconds
0  machining, generic          112.224583  30.525278   109891
1             milling           95.737289  22.623611    81445


# Additional information

#### 7) Calculate total energy consumption per event

In [ ]:
df_totals = (
    final_df
    .reset_index()  # sikrer at company, event_id og Tool_Number er kolonner
    .groupby(["company", "event_id"], as_index=False)
    .agg({
        "energy_consumption": "sum",   
        "Power_Active": "mean",
        "hours": "sum"        
    })
    .set_index(["company", "event_id"])  # hvis du vil have dem som index igen
)

df_meter = (
    pd.DataFrame(event_summaries)
    .set_index(["company", "event_id"])
)

df_totals = df_totals.join(df_meter)

df_totals["rel_diff_calc_vs_meter"] = (
    (df_totals["energy_consumption"] - df_totals["energy_meter_total"])
    / df_totals["energy_meter_total"]
)




In [ ]:
print(df_totals)